# NBS CRM User Clustering

Monthly re-run: extract → preprocess → cluster → profile → export.
Spec: `docs/superpowers/specs/2026-05-18-crm-user-clustering-design.md`

In [ ]:
import os
import warnings
from datetime import datetime

import hdbscan
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore", category=FutureWarning)
load_dotenv()

engine = create_engine(os.environ["READONLY_DATABASE_URL"], pool_pre_ping=True)
ANALYSIS_MONTH = datetime.now().strftime("%Y-%m")
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Analysis month: {ANALYSIS_MONTH}")
print("DB engine ready:", engine.url.host)

In [ ]:
# ── Users base ──────────────────────────────────────────────────────────────
sql_users = text("""
    SELECT
        u.id                                    AS user_id,
        u.kyc_level,
        u.status,
        u.created_at,
        u.last_active_at,
        (u.account_type = 'business')::int      AS account_type_business,
        COALESCE(p.onboarding_completed, FALSE)::int AS onboarding_completed
    FROM users u
    LEFT JOIN user_profiles p ON p.user_id = u.id
    WHERE u.status != 'suspended'
""")

df_users = pd.read_sql(sql_users, engine)
print(f"Users: {len(df_users):,}")
assert df_users["user_id"].nunique() == len(df_users), "Duplicate user_ids"
assert df_users["kyc_level"].between(0, 3).all(), "kyc_level out of range"
df_users.head(3)

In [ ]:
# ── Founders ─────────────────────────────────────────────────────────────────
sql_founders = text("""
    SELECT
        user_id,
        1                   AS is_founder,
        network_size        AS founder_network_size,
        invites_sent
    FROM founders
""")

df_founders = pd.read_sql(sql_founders, engine)
assert df_founders["user_id"].nunique() == len(df_founders), "Duplicate user_ids in founders"
assert (df_founders["founder_network_size"] >= 0).all(), "Negative network_size found"
assert (df_founders["invites_sent"] >= 0).all(), "Negative invites_sent found"
print(f"Founders: {len(df_founders):,}")
df_founders.head(3)

In [ ]:
# ── Onramp / Offramp ─────────────────────────────────────────────────────────
# from_amount_brl is NULL on offramp rows; to_amount_brl is NULL on onramp rows.
# COALESCE(..., 0) before every SUM.
sql_conversions = text("""
    SELECT
        user_id,
        COUNT(*) FILTER (WHERE direction = 'brl_to_usdc')    AS n_onramp_txns,
        COUNT(*) FILTER (WHERE direction = 'usdc_to_brl')    AS n_offramp_txns,
        COALESCE(SUM(
            CASE WHEN direction = 'brl_to_usdc'
                 THEN COALESCE(from_amount_brl, 0) ELSE 0 END
        ) / 100.0, 0)                                        AS total_onramp_brl,
        COALESCE(SUM(
            COALESCE(spread_revenue_brl, 0) + COALESCE(fee_amount_brl, 0)
        ) / 100.0, 0)                                        AS total_spread_revenue_brl,
        COALESCE(
            SUM((processing_mode = 'instant')::int)::float
            / NULLIF(COUNT(*), 0), 0
        )                                                    AS pct_instant_mode,
        MAX(created_at)                                      AS last_conversion_at
    FROM conversion_quotes
    WHERE used = TRUE
    GROUP BY user_id
""")

df_conversions = pd.read_sql(sql_conversions, engine)
print(f"Users with conversions: {len(df_conversions):,}")
assert (df_conversions["total_spread_revenue_brl"] >= 0).all(), "Negative revenue"
assert df_conversions["user_id"].nunique() == len(df_conversions), "Duplicate user_ids"
df_conversions.head(3)

In [ ]:
# ── Cards ────────────────────────────────────────────────────────────────────
sql_cards_txns = text("""
    SELECT
        user_id,
        COUNT(*) FILTER (
            WHERE transaction_type = 'spend' AND status = 'completed'
        )                                   AS n_card_txns,
        COALESCE(SUM(amount) FILTER (
            WHERE transaction_type = 'spend' AND status = 'completed'
        ) / 100.0, 0)                       AS total_card_spend_usd,
        MAX(authorized_at) FILTER (
            WHERE transaction_type = 'spend'
        )                                   AS last_card_spend_at
    FROM card_transactions
    GROUP BY user_id
""")

sql_cards_issued = text("""
    SELECT
        user_id,
        1                                   AS has_card,
        MAX((card_variant = 'founder')::int) AS card_variant_founder
    FROM cards
    WHERE status = 'active'
    GROUP BY user_id
""")

sql_annual_fee = text("""
    SELECT DISTINCT user_id, 1 AS paid_annual_fee
    FROM card_annual_fees
    WHERE status = 'paid'
""")

df_card_txns   = pd.read_sql(sql_cards_txns, engine)
df_card_issued = pd.read_sql(sql_cards_issued, engine)
df_annual_fee  = pd.read_sql(sql_annual_fee, engine)

assert df_annual_fee["user_id"].nunique() == len(df_annual_fee), "Duplicate user_ids in annual_fee"

assert (df_card_txns["total_card_spend_usd"] >= 0).all(), "Negative card spend"
assert df_card_txns["user_id"].nunique() == len(df_card_txns), "Duplicate user_ids in card_txns"
assert df_card_issued["user_id"].nunique() == len(df_card_issued), "Duplicate user_ids in card_issued"

print(f"Users with card txns:   {len(df_card_txns):,}")
print(f"Users with active card: {len(df_card_issued):,}")
print(f"Users paid annual fee:  {len(df_annual_fee):,}")

In [ ]:
# ── Swaps + Solana ────────────────────────────────────────────────────────────
sql_swaps = text("""
    SELECT
        user_id,
        COUNT(*)                                            AS n_swaps,
        COALESCE(SUM(input_amount) / 1e6, 0)               AS total_swap_volume_usdc,
        COUNT(DISTINCT input_mint)
            + COUNT(DISTINCT output_mint)                   AS n_unique_tokens,
        MAX(timestamp)                                      AS last_swap_at
    FROM swap_transactions
    GROUP BY user_id
""")

sql_solana = text("""
    SELECT
        user_id,
        COUNT(DISTINCT transaction_signature) AS n_solana_txns
    FROM solana_sponsored_transactions
    GROUP BY user_id
""")

df_swaps  = pd.read_sql(sql_swaps, engine)
df_solana = pd.read_sql(sql_solana, engine)

assert df_swaps["user_id"].nunique() == len(df_swaps), "Duplicate user_ids in swaps"
assert (df_swaps["total_swap_volume_usdc"] >= 0).all(), "Negative swap volume"
assert df_solana["user_id"].nunique() == len(df_solana), "Duplicate user_ids in solana"

print(f"Users with swaps:  {len(df_swaps):,}")
print(f"Users with Solana: {len(df_solana):,}")

In [ ]:
# ── AI sessions ──────────────────────────────────────────────────────────────
sql_ai = text("""
    SELECT
        user_id,
        COUNT(*)                        AS n_ai_sessions,
        COALESCE(SUM(message_count), 0) AS total_ai_messages
    FROM ai_sessions
    GROUP BY user_id
""")

# ── Notification engagement ───────────────────────────────────────────────
sql_notifications = text("""
    SELECT
        user_id,
        COUNT(*)                                        AS n_notifications_received,
        COUNT(*) FILTER (WHERE read_at IS NOT NULL)     AS n_notifications_read
    FROM notification_events
    GROUP BY user_id
""")

# ── International payouts ─────────────────────────────────────────────────
sql_international = text("""
    SELECT
        user_id,
        COUNT(*) FILTER (WHERE status = 'completed')        AS n_international_payouts,
        COALESCE(SUM(amount) FILTER (WHERE status = 'completed'), 0)
                                                            AS total_international_usdc
    FROM unblockpay_payouts
    GROUP BY user_id
""")

df_ai            = pd.read_sql(sql_ai, engine)
df_notifications = pd.read_sql(sql_notifications, engine)
df_international = pd.read_sql(sql_international, engine)

assert df_ai["user_id"].nunique() == len(df_ai), "Duplicate user_ids in ai"
assert df_notifications["user_id"].nunique() == len(df_notifications), "Duplicate user_ids in notifications"
assert (df_notifications["n_notifications_read"] <= df_notifications["n_notifications_received"]).all(), \
    "Read count exceeds received count"

print(f"Users with AI sessions:   {len(df_ai):,}")
print(f"Users with notifications: {len(df_notifications):,}")
print(f"Users with intl payouts:  {len(df_international):,}")